# 🧠 Weight Decay Optimization and Regularization

Welcome to the hands-on explanation notebook for **Weight Decay**! In this notebook, we will:
1. Explain the math of L2 Regularization, the shrinkage factor $(1 - \alpha \lambda)$, and Decoupled Weight Decay (AdamW).
2. Generate a small noisy dataset and fit a high-degree polynomial regression model.
3. Compare the fitted models with and without L2 regularization (using Ridge regression) to see how weight decay prevents overfitting.
4. Visualize how weight decay shrinks parameter coefficients (weight magnitudes) close to zero.
5. Implement Stochastic Gradient Descent (SGD) with Weight Decay from scratch in Python.
6. Connect these concepts to YOLO's default `weight_decay=0.0005` training parameter.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge

# Set seed for reproducibility
np.random.seed(42)

## 1. Generating Noisy Training Data

We generate a small dataset of 12 noisy training samples drawn from $y = \cos(1.5 \pi x)$. Because the sample size is small, a high-degree polynomial will easily overfit.

In [ ]:
def true_func(x):
    return np.cos(1.5 * np.pi * x)

n_samples = 12
x_train = np.sort(np.random.rand(n_samples))
y_train = true_func(x_train) + np.random.normal(0, 0.15, n_samples)

x_test = np.linspace(0, 1, 100)
y_true = true_func(x_test)

## 2. Fitting Polynomials: Unregularized vs. Regularized (Weight Decay)

Let's fit a Degree 10 polynomial regression using:
1.  **Standard Linear Regression (No Weight Decay):** Captures noise, causing high variance.
2.  **Ridge Regression (Weight Decay / L2 Regularization, $\lambda = 0.05$):** Adds penalty to shrink weight magnitudes.

In [ ]:
degree = 10

# Model 1: No Regularization
model_no_reg = make_pipeline(PolynomialFeatures(degree), LinearRegression())
model_no_reg.fit(x_train[:, np.newaxis], y_train)
y_pred_no_reg = model_no_reg.predict(x_test[:, np.newaxis])

# Model 2: Ridge Regularization
model_reg = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=0.05))
model_reg.fit(x_train[:, np.newaxis], y_train)
y_pred_reg = model_reg.predict(x_test[:, np.newaxis])

# Plotting the comparisons
plt.figure(figsize=(12, 6))
plt.plot(x_test, y_true, color='black', linewidth=2.5, label='True Function f(x)')
plt.plot(x_test, y_pred_no_reg, color='red', linestyle='--', linewidth=2, label='No Weight Decay (Overfitted)')
plt.plot(x_test, y_pred_reg, color='teal', linewidth=3, label='With Weight Decay (Regularized)')
plt.scatter(x_train, y_train, color='orange', edgecolor='k', s=55, zorder=5, label='Training Points')
plt.ylim(-2.5, 2.5)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Weight Decay (L2) Regularization on Degree 10 Polynomial')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

Observe:
-   **Red Line (No Weight Decay):** Bends wildly at the edges to pass exactly through noisy training points, resulting in poor generalisation.
-   **Teal Line (With Weight Decay):** Restricts the weights from blowing up, keeping the curve smooth and highly aligned with the true generating function.

## 3. Visualizing Weight Coefficient Shrinkage

Let's inspect the weights (coefficients) of both models.

In [ ]:
coef_no_reg = np.abs(model_no_reg.named_steps['linearregression'].coef_)
coef_reg = np.abs(model_reg.named_steps['ridge'].coef_)

coef_no_reg = coef_no_reg[1:]
coef_reg = coef_reg[1:]

indices = np.arange(1, len(coef_no_reg) + 1)

plt.figure(figsize=(12, 5))
plt.bar(indices - 0.2, coef_no_reg, width=0.4, color='red', label='No Weight Decay')
plt.bar(indices + 0.2, coef_reg, width=0.4, color='teal', label='With Weight Decay')
plt.yscale('log')
plt.xlabel('Polynomial Term Index')
plt.ylabel('Absolute Coefficient Magnitude (Log Scale)')
plt.title('Effect of Weight Decay on Parameter Coefficients')
plt.xticks(indices)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3, which='both')
plt.show()

Notice how the coefficients for the unregularized model are extremely large, while the regularized model keeps coefficients compressed.

## 4. Implementing Weight Decay from Scratch

Let's write a python function to update weight matrices using SGD with Weight Decay:
$$\mathbf{w}_{t+1} = \mathbf{w}_t(1 - \alpha \lambda) - \alpha \nabla E_0(\mathbf{w}_t)$$

In [ ]:
def update_weights_with_decay(w, grad, lr, weight_decay):
    shrinkage = 1.0 - lr * weight_decay
    w_updated = w * shrinkage - lr * grad
    return w_updated

w_test = np.array([2.5, -1.5])
grad_test = np.array([0.4, -0.2])
print("Updated weights:", update_weights_with_decay(w_test, grad_test, lr=0.1, weight_decay=0.01))

## 💡 Connection to YOLO and Deep Learning
*   **Decoupled Weight Decay (AdamW):** When training YOLO models, you specify `weight_decay=0.0005` in your configurations. Because YOLO uses **AdamW** by default, weight decay is applied directly to the parameters rather than through gradient moving moments.
*   This decoupled shrinkage keeps weights from growing too large, preventing numerical explosions (weights becoming `NaN`) and ensuring the model generalizes well to validation datasets.